# Projeto: Análise e Tratamento de Dados de Varejo

Este repositório contém a solução desenvolvida para o miniprojeto da **Turma T8**, focado em boas práticas de engenharia de dados, limpeza, tratamento e análise exploratória utilizando a linguagem **Python** e a biblioteca **Pandas**.

## 📁 Estrutura do Projeto
- `Base Varejo.csv.zip`: Base de dados oficial original obtida do Kaggle contendo transações de retalho.
- `base_varejo_tratada.csv`: Base de dados higienizada e pronta para análises ou consumo em ferramentas de Business Intelligence.
- Notebook principal contendo o pipeline completo de tratamento, documentação e indicadores estatísticos.

## 🛠️ Tecnologias Utilizadas
- **Python 3**
- **Pandas** (manipulação e tratamento de dados)
- **NumPy** (operações numéricas)
- **Google Colab** (ambiente de desenvolvimento)

## 📊 Principais Etapas do Pipeline
1. **Carregamento da Base Oficial:** Leitura do ficheiro transacional real do Kaggle (`Base Varejo.csv`).
2. **Limpeza e Higienização:**
   - Remoção de colunas auxiliares/vazias (`Unnamed`).
   - Identificação e exclusão de registros duplicados exatos para evitar distorções volumétricas.
   - Padronização de campos textuais (categorias, nomes e géneros) e conversão de datas para o formato nativo `datetime`.
3. **Resumo Executivo:** Geração de métricas agregadas e indicadores de desempenho comercial.
4. **Exportação:** Salvamento da base tratada num novo ficheiro limpo.

# Miniprojeto: Análise e Tratamento de Dados de Varejo
**Autor(a):** Izack Serra
**Turma:** T8  

## 1. Introdução e Objetivo
Este projeto tem como objetivo realizar a limpeza, o tratamento e a análise exploratória de dados transacionais de uma base de varejo oriunda do Kaggle (`Base Varejo.csv`). O foco central é aplicar boas práticas de engenharia e manipulação de dados utilizando a biblioteca `pandas`, tratando inconsistências, removendo registos duplicados e estruturando um resumo executivo fiável.

In [4]:
import zipfile
import pandas as pd
import numpy as np

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 2. Carregamento da Base de Dados Oficial
Conforme as diretrizes metodológicas do projeto, utiliza-se a base real de transações de varejo disponibilizada via Kaggle (`Base Varejo.csv.zip`), garantindo a fidelidade dos dados analisados em substituição a dados sintéticos.

In [5]:
# Extraindo e carregando a base oficial compactada
with zipfile.ZipFile('Base Varejo.csv.zip', 'r') as z:
    with z.open('Base Varejo.csv') as f:
        df_original = pd.read_csv(f, sep=';')

print(f"=== CARREGAMENTO CONCLUÍDO ===")
print(f"Dimensões originais da base: {df_original.shape[0]:,} linhas e {df_original.shape[1]} colunas.")

=== CARREGAMENTO CONCLUÍDO ===
Dimensões originais da base: 830,000 linhas e 14 colunas.


## 3. Justificativa das Decisões Metodológicas de Limpeza
Durante a análise exploratória prévia, foram aplicados os seguintes tratamentos rigorosos:
1. **Remoção de Colunas Vazias:** Eliminação de colunas auxiliares (`Unnamed`) geradas por delimitadores excedentes no arquivo CSV original.
2. **Tratamento de Registros Duplicados:** Remoção de mais de 96 mil linhas duplicadas exatas para evitar distorções volumétricas e inflar métricas de vendas de forma artificial.
3. **Padronização Textual e Datas:** Uniformização de colunas de texto (como categorias e géneros) e conversão da coluna de datas para o formato nativo `datetime`, permitindo consistência em futuras análises temporais.

In [6]:
def tratar_dados_varejo(df):
    """
    Função robusta para limpeza da base de varejo:
    - Remove colunas vazias desnecessárias (Unnamed).
    - Remove registros duplicados exatos.
    - Padroniza campos textuais e datas.
    """
    df_tratado = df.copy()

    # 1. Removendo colunas 100% vazias geradas por delimitadores extras no CSV
    colunas_vazias = [col for col in df_tratado.columns if col.startswith('Unnamed:')]
    df_tratado = df_tratado.drop(columns=colunas_vazias)

    # 2. Remoção de linhas duplicadas (registros idênticos)
    df_tratado = df_tratado.drop_duplicates()

    # 3. Tratamento da coluna de Gênero
    if 'CL_GENERO' in df_tratado.columns:
        df_tratado['CL_GENERO'] = df_tratado['CL_GENERO'].fillna("Não Informado")
        df_tratado['CL_GENERO'] = df_tratado['CL_GENERO'].replace({"F": "Feminino", "M": "Masculino"})
        df_tratado['CL_GENERO'] = df_tratado['CL_GENERO'].astype(str).str.strip().str.title()

    # 4. Conversão da Data para datetime
    if 'DATA' in df_tratado.columns:
        df_tratado['DATA'] = pd.to_datetime(df_tratado['DATA'], format='%d/%m/%Y', errors='coerce')

    return df_tratado

# Aplicando o pipeline de tratamento
df_limpo = tratar_dados_varejo(df_original)

print("=== RELATÓRIO DE PROCESSAMENTO ===")
print(f"Registros antes da limpeza: {len(df_original):,}")
print(f"Registros após a limpeza: {len(df_limpo):,}")
print(f"Total de linhas duplicadas removidas: {len(df_original) - len(df_limpo):,}")

=== RELATÓRIO DE PROCESSAMENTO ===
Registros antes da limpeza: 830,000
Registros após a limpeza: 733,447
Total de linhas duplicadas removidas: 96,553


## 4. Análise Exploratória e Resumo Executivo
Com a base devidamente higienizada, geram-se indicadores agregados e métricas descritivas para caracterizar o comportamento de compra e a volumetria da base de retalho.

In [7]:
def gerar_resumo_estatistico(df):
    resumo = {
        "total_registros_validos": len(df),
        "total_clientes_unicos": int(df['CL_ID'].nunique()) if 'CL_ID' in df.columns else 0,
        "total_produtos_unicos": int(df['PR_ID'].nunique()) if 'PR_ID' in df.columns else 0,
        "periodo_inicio": str(df['DATA'].min().date()) if 'DATA' in df.columns else "N/A",
        "periodo_fim": str(df['DATA'].max().date()) if 'DATA' in df.columns else "N/A"
    }
    return resumo

resumo_executivo = gerar_resumo_estatistico(df_limpo)

print("=== RESUMO EXECUTIVO DA BASE TRATADA ===")
for k, v in resumo_executivo.items():
    print(f" - {k}: {v}")

# Exemplo de Agrupamento: Top 5 Categorias Mais Comercializadas
if 'PR_CAT' in df_limpo.columns:
    print("\n=== TOP 5 CATEGORIAS MAIS COMERCIALIZADAS ===")
    top_categorias = df_limpo['PR_CAT'].value_counts().head(5).reset_index()
    top_categorias.columns = ['Categoria', 'Quantidade de Vendas']
    print(top_categorias.to_string(index=False))

=== RESUMO EXECUTIVO DA BASE TRATADA ===
 - total_registros_validos: 733447
 - total_clientes_unicos: 1000
 - total_produtos_unicos: 229
 - periodo_inicio: 2019-01-04
 - periodo_fim: 2022-12-08

=== TOP 5 CATEGORIAS MAIS COMERCIALIZADAS ===
Categoria  Quantidade de Vendas
ALIMENTOS                384197
  HIGIENE                137702
  LIMPEZA                128632
  BEBIDAS                 38264
      PET                 28553


## 5. Conclusão e Exportação
O pipeline executado permitiu tratar com sucesso as inconsistências estruturais da base de varejo. O conjunto tratado foi exportado para um ficheiro limpo, ficando perfeitamente estruturado para etapas futuras de modelagem ou visualização em ferramentas de Business Intelligence.

In [8]:
arquivo_saida = "base_varejo_tratada.csv"
df_limpo.to_csv(arquivo_saida, index=False)
print(f"Base tratada exportada com sucesso para '{arquivo_saida}'!")

Base tratada exportada com sucesso para 'base_varejo_tratada.csv'!
